In [ ]:
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("../data/obesity_challenge_2.h5ad")

# -----------------------
# 1. 只看 control cells
# -----------------------

adata_ctrl = adata[
    adata.obs["gene"].isin(["NC", "NC+NC"])
].copy()

# -----------------------
# 2. 建立 state label
# -----------------------

def assign_state_label(df):

    labels = []

    for _, r in df.iterrows():

        if (r["lipo"] == 1) and (r["adipo"] == 1):
            labels.append("lipo_adipo")

        elif r["lipo"] == 1:
            labels.append("lipo")

        elif r["adipo"] == 1:
            labels.append("adipo")

        elif r["pre_adipo"] == 1:
            labels.append("pre_adipo")

        else:
            labels.append("other")

    return pd.Series(labels, index=df.index)


adata_ctrl.obs["state"] = assign_state_label(
    adata_ctrl.obs
)

# -----------------------
# 3. overall state distribution
# -----------------------

print("\n===== STATE distribution =====\n")

state_dist = (
    adata_ctrl.obs["state"]
    .value_counts()
    .sort_index()
)

print(state_dist)

print("\nproportion:")
print(
    (state_dist / state_dist.sum())
    .round(3)
)

# -----------------------
# 4. state x batch table
# -----------------------

print("\n===== STATE x BATCH =====\n")

state_batch_table = pd.crosstab(

    adata_ctrl.obs["state"],
    adata_ctrl.obs["SampleID"]

)

print(state_batch_table)

# -----------------------
# 5. normalized version
# -----------------------

print("\n===== STATE x BATCH proportion =====\n")

state_batch_prop = (

    state_batch_table
    .div(
        state_batch_table.sum(axis=1),
        axis=0
    )
    .round(3)

)

print(state_batch_prop)

# -----------------------
# 6. per state batch diversity summary
# -----------------------

print("\n===== batches per state =====\n")

batch_counts_per_state = (

    adata_ctrl.obs
    .groupby("state")["SampleID"]
    .nunique()
)

print(batch_counts_per_state)

# -----------------------
# 7. optional save
# -----------------------

# state_batch_table.to_csv(
#     "../data/state_batch_table_control.csv"
# )


===== STATE distribution =====

state
adipo         1674
lipo_adipo     578
other         4051
pre_adipo     2357
Name: count, dtype: int64

proportion:
state
adipo         0.193
lipo_adipo    0.067
other         0.468
pre_adipo     0.272
Name: count, dtype: float64

===== STATE x BATCH =====

SampleID    TF15_MOI2_1  TF15_MOI2_2  TF15_MOI2_3  TF15_MOI2_4  TF15_MOI2_5  \
state                                                                         
adipo               147          126          159          149          175   
lipo_adipo           47           54           59           54           43   
other               320          317          261          260          297   
pre_adipo           221          238          233          242          229   

SampleID    TF15_MOI2_6  TF15_MOI2_7  TF15_MOI2_8  TF15_MOI2_9  TF15_MOI2_10  \
state                                                                          
adipo               165          144          146          134       

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

np.random.seed(42)

# =====================================================
# CONFIG
# =====================================================

DATA_PATH = "../data/obesity_challenge_2.h5ad"
SAVE_PATH = "../data/preprocessed/control_cells/control_cells_all.h5ad"

N_CONTROL_NC = 100

STATE_PROP = {
    "adipo": 0.193,
    "pre_adipo": 0.272,
    "other": 0.467,
    "lipo_adipo": 0.067
}

# =====================================================
# LOAD DATA
# =====================================================

adata = sc.read_h5ad(DATA_PATH)

adata_nc = adata[adata.obs["gene"] == "NC"].copy()
adata_nc_nc = adata[adata.obs["gene"] == "NC+NC"].copy()

batches = sorted(adata_nc.obs["SampleID"].unique())

# =====================================================
# DEFINE CELL STATE
# =====================================================

def assign_state_label(df):

    labels = []

    for _, r in df.iterrows():

        if (r["lipo"] == 1) and (r["adipo"] == 1):
            labels.append("lipo_adipo")

        elif r["lipo"] == 1:
            labels.append("lipo")

        elif r["adipo"] == 1:
            labels.append("adipo")

        elif r["pre_adipo"] == 1:
            labels.append("pre_adipo")

        else:
            labels.append("other")

    return pd.Series(labels, index=df.index)


adata_nc.obs["state"] = assign_state_label(adata_nc.obs)
adata_nc_nc.obs["state"] = assign_state_label(adata_nc_nc.obs)

# =====================================================
# BATCH TARGET (8–9 per batch)
# =====================================================

def compute_batch_targets(batches, total_cells):

    base = total_cells // len(batches)

    batch_target = {b: base for b in batches}

    remainder = total_cells - base * len(batches)

    extra_batches = np.random.choice(
        batches,
        size=remainder,
        replace=False
    )

    for b in extra_batches:
        batch_target[b] += 1

    return batch_target


batch_target = compute_batch_targets(
    batches,
    N_CONTROL_NC
)

# =====================================================
# SAMPLING FUNCTION
# =====================================================

def sample_nc_cells(adata_nc, batches, batch_target):

    selected_cells = []

    for batch in batches:

        batch_cells = adata_nc[
            adata_nc.obs["SampleID"] == batch
        ]

        n_batch = batch_target[batch]

        chosen = []

        # ------------------
        # ensure 1 lipo_adipo
        # ------------------

        lipo_cells = batch_cells[
            batch_cells.obs["state"] == "lipo_adipo"
        ]

        if len(lipo_cells) > 0:

            chosen_lipo = np.random.choice(
                lipo_cells.obs_names,
                size=1,
                replace=False
            )

            chosen.extend(chosen_lipo)

        remaining_n = n_batch - len(chosen)

        # ------------------
        # proportional sampling
        # ------------------

        pool = batch_cells[
            ~batch_cells.obs_names.isin(chosen)
        ]

        state_counts = {

            s: int(round(STATE_PROP[s] * remaining_n))
            for s in STATE_PROP
            if s != "lipo_adipo"
        }

        diff = remaining_n - sum(state_counts.values())

        if diff > 0:

            ordered_states = sorted(
                state_counts,
                key=lambda s: STATE_PROP[s],
                reverse=True
            )

            for i in range(diff):

                state_counts[
                    ordered_states[i % len(ordered_states)]
                ] += 1

        # sample each state
        for state, n_pick in state_counts.items():

            state_cells = pool[
                pool.obs["state"] == state
            ]

            if len(state_cells) == 0:
                continue

            n_pick = min(n_pick, len(state_cells))

            sampled = np.random.choice(
                state_cells.obs_names,
                size=n_pick,
                replace=False
            )

            chosen.extend(sampled)

        # fill remaining randomly if needed
        remaining_pool = batch_cells[
            ~batch_cells.obs_names.isin(chosen)
        ]

        if len(chosen) < n_batch:

            extra = np.random.choice(
                remaining_pool.obs_names,
                size=n_batch - len(chosen),
                replace=False
            )

            chosen.extend(extra)

        selected_cells.extend(chosen)

    return selected_cells


# =====================================================
# RUN SAMPLING
# =====================================================

selected_nc_cells = sample_nc_cells(

    adata_nc,
    batches,
    batch_target

)

adata_nc_selected = adata_nc[selected_nc_cells].copy()

# =====================================================
# COMBINE NC + NC+NC
# =====================================================

adata_control = sc.concat(
    [adata_nc_selected, adata_nc_nc],
    join="outer"
)

# =====================================================
# DIAGNOSTICS
# =====================================================

def print_diagnostics(adata_subset, label):

    print("\n====================")
    print(label)
    print("====================")

    print("\nSTATE distribution")
    print(
        adata_subset.obs["state"]
        .value_counts()
    )

    print("\nSTATE proportion")
    print(
        adata_subset.obs["state"]
        .value_counts(normalize=True)
        .round(3)
    )

    print("\nBATCH distribution")
    print(
        adata_subset.obs["SampleID"]
        .value_counts()
    )


adata_nc_only = adata_control[
    adata_control.obs["gene"] == "NC"
]

adata_nc_nc_only = adata_control[
    adata_control.obs["gene"] == "NC+NC"
]

print_diagnostics(adata_nc_only, "NC only")

print_diagnostics(adata_nc_nc_only, "NC+NC only")

# pivot tables

print("\n====================")
print("NC state x batch")
print("====================")

print(
    pd.crosstab(
        adata_nc_only.obs["state"],
        adata_nc_only.obs["SampleID"]
    )
)

print("\n====================")
print("NC+NC state x batch")
print("====================")

print(
    pd.crosstab(
        adata_nc_nc_only.obs["state"],
        adata_nc_nc_only.obs["SampleID"]
    )
)

# =====================================================
# SAVE
# =====================================================

adata_control.write_h5ad(SAVE_PATH)

print("\nSaved:")
print(SAVE_PATH)


NC only

STATE distribution
state
other         48
pre_adipo     24
adipo         16
lipo_adipo    12
Name: count, dtype: int64

STATE proportion
state
other         0.48
pre_adipo     0.24
adipo         0.16
lipo_adipo    0.12
Name: proportion, dtype: float64

BATCH distribution
SampleID
TF15_MOI2_1     9
TF15_MOI2_6     9
TF15_MOI2_7     9
TF15_MOI2_8     9
TF15_MOI2_2     8
TF15_MOI2_3     8
TF15_MOI2_4     8
TF15_MOI2_5     8
TF15_MOI2_9     8
TF15_MOI2_10    8
TF15_MOI2_11    8
TF15_MOI2_12    8
Name: count, dtype: int64

NC+NC only

STATE distribution
state
other         10
pre_adipo      8
adipo          7
lipo_adipo     3
Name: count, dtype: int64

STATE proportion
state
other         0.357
pre_adipo     0.286
adipo         0.250
lipo_adipo    0.107
Name: proportion, dtype: float64

BATCH distribution
SampleID
TF15_MOI2_6     5
TF15_MOI2_2     4
TF15_MOI2_7     4
TF15_MOI2_4     3
TF15_MOI2_10    3
TF15_MOI2_3     2
TF15_MOI2_5     2
TF15_MOI2_9     2
TF15_MOI2_12    2
TF15_MO

In [ ]:
# =====================================================
# extract NC only
# =====================================================

adata_nc_only = adata_control[
    adata_control.obs["gene"] == "NC"
].copy()

NC_SAVE_PATH = "../data/preprocessed/control_cells/NC_control_cells.h5ad"

adata_nc_only.write_h5ad(NC_SAVE_PATH)

print("\nSaved NC control cells:")
print(NC_SAVE_PATH)

print("\nNC cell count:")
print(adata_nc_only.n_obs)


Saved NC control cells:
../data/preprocessed/NC_control_cells.h5ad

NC cell count:
100


In [13]:
def replace_nc_with_nc_nc(adata_control):
    
    """
    Replace NC cells with NC+NC cells
    matching SAME state AND SAME batch.

    Returns
    -------
    adata_replaced
    replacement_table
    summary stats
    """

    import pandas as pd

    nc_cells = adata_control[
        adata_control.obs["gene"] == "NC"
    ].copy()

    nc_nc_cells = adata_control[
        adata_control.obs["gene"] == "NC+NC"
    ].copy()

    used_nc_nc = set()

    replacements = []

    for nc_idx, nc_row in nc_cells.obs.iterrows():

        state = nc_row["state"]
        batch = nc_row["SampleID"]

        candidates = nc_nc_cells[
            (nc_nc_cells.obs["state"] == state)
            &
            (nc_nc_cells.obs["SampleID"] == batch)
            &
            (~nc_nc_cells.obs_names.isin(used_nc_nc))
        ]

        if len(candidates) > 0:

            chosen_nc_nc = candidates.obs_names[0]

            replacements.append({

                "NC_cell": nc_idx,
                "NCNC_cell": chosen_nc_nc,
                "state": state,
                "batch": batch

            })

            used_nc_nc.add(chosen_nc_nc)

    replacement_df = pd.DataFrame(replacements)

    # apply replacement
    nc_to_remove = replacement_df["NC_cell"].tolist()

    nc_nc_to_add = replacement_df["NCNC_cell"].tolist()

    nc_remaining = nc_cells[
        ~nc_cells.obs_names.isin(nc_to_remove)
    ]

    nc_nc_used = nc_nc_cells[
        nc_nc_cells.obs_names.isin(nc_nc_to_add)
    ]

    adata_replaced = sc.concat([

        nc_remaining,
        nc_nc_used

    ])

    # summary
    print("\n====================")
    print("Replacement summary")
    print("====================")

    print("\nTotal NC replaced:")
    print(len(replacement_df))

    print("\nRemaining NC:")
    print(len(nc_remaining))

    print("\nNC+NC used:")
    print(len(nc_nc_used))

    print("\nReplacements per state:")

    print(
        replacement_df["state"]
        .value_counts()
    )

    print("\nReplacements per batch:")

    print(
        replacement_df["batch"]
        .value_counts()
    )

    return adata_replaced, replacement_df

In [ ]:
SAVE_PATH = "../data/preprocessed/control_cells/NC_NC_control_cells.h5ad"

adata_replaced, replacement_table = replace_nc_with_nc_nc(adata_control)

print("\nReplacement table preview:")
print(replacement_table.head())
print_diagnostics(adata_replaced, "NC Replacement")

adata_replaced.write_h5ad(SAVE_PATH)

print("\nSaved:")
print(SAVE_PATH)



Replacement summary

Total NC replaced:
27

Remaining NC:
73

NC+NC used:
27

Replacements per state:
state
other         10
pre_adipo      8
adipo          6
lipo_adipo     3
Name: count, dtype: int64

Replacements per batch:
batch
TF15_MOI2_6     5
TF15_MOI2_7     4
TF15_MOI2_10    3
TF15_MOI2_2     3
TF15_MOI2_4     3
TF15_MOI2_12    2
TF15_MOI2_3     2
TF15_MOI2_5     2
TF15_MOI2_9     2
TF15_MOI2_1     1
Name: count, dtype: int64

Replacement table preview:
                  NC_cell               NCNC_cell       state         batch
0   P1_AACTGCATCCTCTCTC-1   P1_GTGTTACGTCTAAGGA-1  lipo_adipo   TF15_MOI2_1
1  P10_GGGTCAGGTGAACTGA-1  P10_CTTAGGATCATGCGCG-1       adipo  TF15_MOI2_10
2  P10_CGCACGATCCCTCGGT-1  P10_ATGCTAACAACATTAG-1   pre_adipo  TF15_MOI2_10
3  P10_ACCATTTCAGGGCTGT-1  P10_AAATGCTTCTGGGCTG-1       other  TF15_MOI2_10
4  P12_CAAGCCACATCCTAAT-1  P12_TCACCCGCAATCATCT-1       adipo  TF15_MOI2_12

NC Replacement

STATE distribution
state
other         48
pre_adipo     24
